# Resultados cuantizados: varias corridas, varianza e intervalos de confianza

Se le da una carpeta (`RESULTS_DIR`) y busca **todos** los `raw/*.json` que haya debajo, a cualquier profundidad.
Cada carpeta que contiene un `raw/` se toma como **una corrida** (ej. `results_quantized`, `results_quantized-run1`, ...).

Del **nombre de cada archivo** salen el modelo y el método:

| archivo | modelo | método |
|---|---|---|
| `meta-llama_Llama-3.1-8B.json` | Llama-3.1-8B | baseline |
| `..._quantized_Llama-3.1-8B-awq.json` | Llama-3.1-8B | AWQ |
| `..._quantized_Llama-3.1-8B-rtn.json` | Llama-3.1-8B | RTN |
| `pretrained_meta-llama_Llama-3.1-8B_load_in_4bit_True_..._nf4_....json` | Llama-3.1-8B | bnb NF4 |

Si la carpeta tiene más de un modelo, cada gráfico se hace **por modelo**, con el modelo en el título.

**Hay dos fuentes de incertidumbre distintas, y cada una se trata por separado:**

1. **Accuracy (calidad del modelo).** La evaluación es determinista (greedy), así que la accuracy sale *idéntica*
   en todas las corridas (la sección 2 lo verifica). La incertidumbre real viene de que el benchmark es una
   **muestra finita de preguntas**. lm-eval reporta el error estándar binomial (`stderr` $=\sqrt{p(1-p)/n}$), y el
   IC 95 % es $\hat p \pm 1.96\,\mathrm{SE}$.
2. **Tiempo / throughput / memoria (rendimiento).** Esto sí varía entre corridas. Con $k$ corridas:
   $\bar x \pm t_{0.975,\,k-1}\, s/\sqrt{k}$ (t de Student). Con pocas corridas el factor $t$ es grande
   (12.7 con k=2, 4.30 con k=3), así que los IC salen anchos.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======================= CONFIGURACIÓN =======================
# Carpeta donde buscar. Se buscan todos los **/raw/*.json debajo de ella.
# Puede ser relativa a la carpeta de la notebook o a la raíz del repo.
RESULTS_DIR = "raw_results/Atom/quantized"
# Tareas a analizar. Todo lo que no esté en esta lista se ignora.
TASKS = ["boolq", "wic", "openbookqa", "sqac"]
# =============================================================

try:
    from scipy import stats
    def t_crit(df, conf=0.95):
        return float(stats.t.ppf(0.5 + conf / 2, df))
except ImportError:  # fallback sin scipy (solo IC 95 %)
    _T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228}
    def t_crit(df, conf=0.95):
        assert conf == 0.95, "sin scipy solo está soportado conf=0.95"
        return _T95.get(df, 1.96)

Z95 = 1.96

def resolve_dir(d):
    # Busca la carpeta relativa al cwd y a cada carpeta padre (así funciona desde plots/ o desde la raíz)
    d = Path(d).expanduser()
    if d.is_absolute():
        return d
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / d).is_dir():
            return (base / d).resolve()
    raise FileNotFoundError(f"No encuentro la carpeta {d} desde {Path.cwd()} ni desde sus padres")

results_dir = resolve_dir(RESULTS_DIR)
raw_files = sorted(results_dir.rglob("raw/*.json"))
print("Carpeta:", results_dir)
print(f"{len(raw_files)} archivos raw encontrados")
assert raw_files, "No hay ningún raw/*.json debajo de RESULTS_DIR"

# Orden y color fijos por método (el color sigue al método, no al orden en el gráfico)
METHOD_ORDER = ["baseline", "AWQ", "bnb NF4", "RTN"]
METHOD_COLORS = {
    "baseline": "#2a78d6",
    "AWQ":      "#eb6834",
    "bnb NF4":  "#eda100",
    "RTN":      "#1baf7a",
}
EXTRA_COLORS = ["#e87ba4", "#008300", "#8a63d2", "#7b7b7b"]   # para métodos que no estén arriba

# Métrica "principal" de cada benchmark
SCORE_METRIC = {"sqac": "f1"}   # el resto usa "acc"

## 1. Lectura de los archivos: modelo y método salen del nombre

In [ ]:
# sufijo "-<método>" al final del nombre del modelo
SUFFIX_METHODS = {"awq": "AWQ", "rtn": "RTN", "gptq": "GPTQ", "nf4": "bnb NF4", "fp4": "bnb FP4",
                  "int8": "INT8", "int4": "INT4", "fp8": "FP8", "gguf": "GGUF", "hqq": "HQQ"}

def parse_name(path: Path):
    # Devuelve (modelo, método) a partir del nombre del archivo
    stem = path.stem
    m = re.match(r"pretrained_(.+?)_load_in_(4|8)bit", stem)
    if m:  # bitsandbytes: pretrained_<org>_<modelo>_load_in_4bit_True_..._quant_type_nf4_...
        model = m.group(1).split("_", 1)[-1]          # meta-llama_Llama-3.1-8B -> Llama-3.1-8B
        if m.group(2) == "8":
            return model, "bnb INT8"
        qt = re.search(r"quant_type_([A-Za-z0-9]+)", stem)
        return model, f"bnb {(qt.group(1) if qt else 'fp4').upper()}"
    name = stem.rsplit("_", 1)[-1]                     # ..._quantized_Llama-3.1-8B-awq -> Llama-3.1-8B-awq
    base, _, suffix = name.rpartition("-")
    if base and suffix.lower() in SUFFIX_METHODS:
        return base, SUFFIX_METHODS[suffix.lower()]
    return name, "baseline"                            # meta-llama_Llama-3.1-8B -> Llama-3.1-8B

def to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):   # "N/A", None
        return np.nan

PERF_KEYS = ["load_time_s", "ttft_ms", "prefill_tok_per_s", "decode_tok_per_s", "e2e_tok_per_s",
             "load_peak_mem_device_gb", "perf_peak_mem_alloc_gb", "perf_peak_mem_device_gb"]

file_rows, score_rows, perf_rows, time_rows = [], [], [], []
for path in raw_files:
    run_dir = path.parent.parent                       # la carpeta que contiene raw/
    run = str(run_dir.relative_to(results_dir)) if run_dir != results_dir else results_dir.name
    model, method = parse_name(path)
    data = json.loads(path.read_text())
    tasks_done = data.get("tasks_done") or sorted({r["task"] for r in data["rows"]})
    tasks_done = [t for t in tasks_done if t in TASKS]
    file_rows.append({"model": model, "method": method, "run": run, "tasks": tasks_done,
                      "perf": bool(data.get("perf")), "file": path.name})

    sample_len = {r["task"]: r["value"] for r in data["rows"] if r["metric"] == "sample_len"}
    seen = set()
    for r in data["rows"]:
        task = r["task"]
        if task not in TASKS:
            continue
        if task not in seen:
            seen.add(task)
            time_rows.append({"model": model, "method": method, "run": run, "task": task,
                              "total_time_s": to_float(r.get("total_time_s"))})
        if r["metric"] == "sample_len":
            continue
        n = r.get("n_samples") or sample_len.get(task)
        score_rows.append({"model": model, "method": method, "run": run, "task": task,
                           "metric": r["metric"], "value": to_float(r["value"]),
                           "stderr": to_float(r.get("stderr")), "n": int(n) if n else np.nan})
    perf = data.get("perf") or {}
    perf_rows.append({"model": model, "method": method, "run": run,
                      **{k: to_float(perf.get(k)) for k in PERF_KEYS}})

files_df = pd.DataFrame(file_rows)
scores = pd.DataFrame(score_rows)
perf_df = pd.DataFrame(perf_rows)
times = pd.DataFrame(time_rows)

MODELS = sorted(files_df["model"].unique())
RUNS = sorted(files_df["run"].unique())
methods_found = sorted(set(files_df["method"]) - set(METHOD_ORDER))
METHODS = METHOD_ORDER + methods_found
for i, m in enumerate(methods_found):
    METHOD_COLORS[m] = EXTRA_COLORS[i % len(EXTRA_COLORS)]

def method_cat(s):
    return pd.Categorical(s, METHODS, ordered=True)

print("Modelos:", MODELS)
print("Corridas:", RUNS)
print("Tareas:", TASKS)
files_df[["model", "method", "run", "file"]].sort_values(["model", "method", "run"]).reset_index(drop=True)

## 2. ¿Qué hay en la carpeta?

Por modelo: si está cada método (baseline, AWQ, bnb, RTN) y qué tareas corrió.
Cada celda dice **en cuántas corridas** está esa tarea, sobre el total de corridas en las que aparece el modelo
(`3/3` = completo, `—` = nunca se corrió).

In [ ]:
all_tasks = TASKS

def coverage_table(model):
    sub = files_df[files_df["model"] == model]
    n_runs = sub["run"].nunique()          # corridas en las que aparece este modelo
    rows = []
    for method in METHODS:
        msub = sub[sub["method"] == method]
        if msub.empty and method not in METHOD_ORDER:
            continue
        row = {"método": method,
               "está": "sí" if len(msub) else "NO",
               "corridas": f"{len(msub)}/{n_runs}",
               "perf": f"{int(msub['perf'].sum())}/{n_runs}"}
        for t in all_tasks:
            k = sum(t in ts for ts in msub["tasks"])
            row[t] = f"{k}/{n_runs}" if k else "—"
        rows.append(row)
    return pd.DataFrame(rows).set_index("método"), n_runs

def highlight(v):
    if v in ("NO", "—"):
        return "background-color: #f8d7da; color: #721c24"
    if isinstance(v, str) and "/" in v:
        a, b = v.split("/")
        return "" if a == b else "background-color: #fff3cd; color: #856404"
    return ""

for model in MODELS:
    tbl, n_runs = coverage_table(model)
    display(tbl.style.map(highlight).set_caption(f"{model}: métodos y tareas presentes ({n_runs} corridas)"))

In [ ]:
# Detalle por corrida: qué tareas tiene cada archivo (✓ / ✗)
detail = files_df.copy()
for t in all_tasks:
    detail[t] = detail["tasks"].apply(lambda ts, t=t: "✓" if t in ts else "✗")
detail["method"] = method_cat(detail["method"])
detail.sort_values(["model", "method", "run"])[["model", "method", "run", "perf", *all_tasks]].reset_index(drop=True)

## 3. ¿Cambia la accuracy entre corridas?

Si `std_entre_corridas` es 0, la evaluación es determinista y repetir corridas **no** aporta información sobre la
calidad. Entonces el IC tiene que salir de la variabilidad entre preguntas (sección 4).

In [ ]:
run_var = (scores.groupby(["model", "method", "task", "metric"], observed=True)["value"]
           .agg(n_corridas="count", media="mean", std_entre_corridas="std", min="min", max="max")
           .reset_index())
run_var["method"] = method_cat(run_var["method"])
run_var = run_var.sort_values(["model", "task", "metric", "method"]).reset_index(drop=True)
print("Máximo desvío entre corridas:", run_var["std_entre_corridas"].max())
run_var

## 4. IC 95 % de la accuracy (variabilidad por muestreo de preguntas)

- Si lm-eval da `stderr`, se usa ese.
- Si no (las métricas de `sqac`) y la métrica es 0/1 por pregunta (`exact_match`, `contains`), se usa el SE binomial $\sqrt{p(1-p)/n}$.
- Para el `f1` de sqac no se puede sacar el SE solo con el promedio: haría falta el F1 **de cada pregunta**
  para hacer *bootstrap*. Por ahora queda sin IC.

In [ ]:
BINARY_METRICS = {"acc", "acc_norm", "exact_match", "contains"}

def se_for(row):
    if not np.isnan(row["stderr"]):
        return row["stderr"]
    if row["metric"] in BINARY_METRICS and not np.isnan(row["n"]):
        p = row["value"]
        return math.sqrt(p * (1 - p) / row["n"])
    return np.nan

# Una fila por (modelo, método, tarea, métrica). El valor no cambia entre corridas, así que se toma la media
acc = (scores.groupby(["model", "method", "task", "metric"])
       .agg(value=("value", "mean"), stderr=("stderr", "mean"), n=("n", "max"), n_corridas=("run", "count"))
       .reset_index())
acc["se"] = acc.apply(se_for, axis=1)
acc["ci_half"] = Z95 * acc["se"]
acc["ci_low"] = acc["value"] - acc["ci_half"]
acc["ci_high"] = acc["value"] + acc["ci_half"]
acc["method"] = method_cat(acc["method"])
acc = acc.sort_values(["model", "task", "metric", "method"]).reset_index(drop=True)

# Métrica principal de cada benchmark
main = acc[acc.apply(lambda r: r["metric"] == SCORE_METRIC.get(r["task"], "acc"), axis=1)].copy()
main["bench"] = main["task"] + " (" + main["metric"] + ")"
acc.round(4)

In [ ]:
# Tabla compacta "valor ± IC95" por modelo (lista para la tesis)
def fmt(r):
    return f"{r['value']:.3f}" if np.isnan(r["ci_half"]) else f"{r['value']:.3f} ± {r['ci_half']:.3f}"

main["valor ± IC95"] = main.apply(fmt, axis=1)
for model in MODELS:
    t = (main[main["model"] == model]
         .pivot(index="method", columns="bench", values="valor ± IC95")
         .dropna(how="all"))
    display(t.style.set_caption(f"{model}: score ± IC 95 %"))

In [ ]:
def style_ax(ax, grid_axis="y"):
    ax.grid(axis=grid_axis, alpha=0.3)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

def bottom_legend(ax, n, y=-0.1):
    ax.legend(title="Método", loc="upper center", bbox_to_anchor=(0.5, y), ncol=max(n, 1), frameon=False)

def grouped_bars(ax, df, group_col, value_col, err_col, groups, ylabel, title, ylim=None, fmt="{:.2f}", log=False):
    # Barras agrupadas: un grupo por `group_col`, una barra por método, barra de error = IC 95 %
    methods = [m for m in METHODS if m in set(df["method"])]
    x = np.arange(len(groups))
    width = 0.8 / max(len(methods), 1)
    for i, m in enumerate(methods):
        sub = df[df["method"] == m].set_index(group_col).reindex(groups)
        vals = sub[value_col].astype(float).values
        errs = np.nan_to_num(sub[err_col].astype(float).values)
        offset = (i - (len(methods) - 1) / 2) * width
        lower = np.minimum(errs, vals * 0.95) if log else errs   # en escala log no se puede bajar de 0
        ax.bar(x + offset, vals, width * 0.92, label=m, color=METHOD_COLORS[m])
        ax.errorbar(x + offset, vals, yerr=[lower, errs], fmt="none", ecolor="#333333", elinewidth=1.2, capsize=3)
        top = ylim[1] if ylim else np.nanmax(df[value_col].astype(float))
        for xi, v, e in zip(x + offset, vals, errs):
            if not np.isnan(v):
                y_txt = (v + e) * 1.08 if log else v + e + top * 0.01
                ax.text(xi, y_txt, fmt.format(v), ha="center", va="bottom", fontsize=7, color="#333333")
    if log:
        ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(groups)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    style_ax(ax)
    bottom_legend(ax, len(methods))

for model in MODELS:
    d = main[main["model"] == model]
    benches = sorted(d["bench"].unique())
    fig, ax = plt.subplots(figsize=(11, 5.5))
    grouped_bars(ax, d, "bench", "value", "ci_half", benches, "Score",
                 f"{model}: score por benchmark y método (barras = IC 95 %)", ylim=(0, 1))
    plt.tight_layout()
    plt.show()

### Forest plot: diferencia contra el baseline del mismo modelo

$\Delta = \text{score}_{\text{cuant}} - \text{score}_{\text{baseline}}$ con su IC. Si el IC **cruza el 0**, la diferencia
no es estadísticamente significativa con este tamaño de benchmark.

Aproximación: $\mathrm{SE}_\Delta = \sqrt{\mathrm{SE}_1^2 + \mathrm{SE}_2^2}$. Como los dos modelos responden las *mismas*
preguntas, esta fórmula es conservadora: el IC pareado real es más angosto. Con las predicciones de cada pregunta se
podría hacer McNemar o un bootstrap pareado.

In [ ]:
diffs = []
for model in MODELS:
    d = main[main["model"] == model]
    base = d[d["method"] == "baseline"].set_index("bench")
    q = d[(d["method"] != "baseline") & d["bench"].isin(base.index)].copy()
    if base.empty or q.empty:
        print(f"{model}: no hay baseline (o no hay cuantizados) para comparar")
        continue
    q["delta"] = q["value"].values - base.loc[q["bench"], "value"].values
    q["se_delta"] = np.sqrt(q["se"].values ** 2 + base.loc[q["bench"], "se"].values ** 2)
    q["delta_ci_half"] = Z95 * q["se_delta"]
    q["significativo"] = np.where(q["se_delta"].isna(), "sin IC", np.where(q["delta"].abs() > q["delta_ci_half"], "sí", "no"))
    diffs.append(q)

    benches_d = sorted(q["bench"].unique())
    methods_d = [m for m in METHODS if m in set(q["method"])]
    fig, ax = plt.subplots(figsize=(9, 1.0 + 0.45 * len(benches_d) * len(methods_d)))
    y = np.arange(len(benches_d))
    h = 0.8 / len(methods_d)
    for i, m in enumerate(methods_d):
        sub = q[q["method"] == m].set_index("bench").reindex(benches_d)
        yy = y + (i - (len(methods_d) - 1) / 2) * h
        ax.errorbar(sub["delta"], yy, xerr=np.nan_to_num(sub["delta_ci_half"]), fmt="o", ms=8,
                    color=METHOD_COLORS[m], ecolor=METHOD_COLORS[m], elinewidth=2, capsize=4,
                    markeredgecolor="white", markeredgewidth=1.5, label=m)
    ax.axvline(0, color="#555555", lw=1, ls="--")
    ax.set_yticks(y)
    ax.set_yticklabels(benches_d)
    ax.invert_yaxis()
    ax.set_xlabel("Δ score vs baseline (IC 95 %)")
    ax.set_title(f"{model}: diferencia contra el baseline (si cruza 0, no es significativa)")
    style_ax(ax, "x")
    bottom_legend(ax, len(methods_d), y=-0.12)
    plt.tight_layout()
    plt.show()

if diffs:
    diff = pd.concat(diffs)
    display(diff[["model", "bench", "method", "delta", "delta_ci_half", "significativo"]].round(4).reset_index(drop=True))

## 5. Rendimiento: varianza entre corridas e IC con t de Student

Para cada métrica: media, desvío estándar (`std`, ddof=1), coeficiente de variación (`cv_% = std/media`) e
IC 95 % $= \bar x \pm t_{0.975,k-1}\, s/\sqrt{k}$. Con una sola corrida no se puede estimar la varianza, y el IC queda vacío.

In [ ]:
def summarize(df, keys, value_cols):
    long = df.melt(id_vars=keys + ["run"], value_vars=value_cols, var_name="metric", value_name="x")
    long = long.dropna(subset=["x"])
    out = long.groupby(keys + ["metric"])["x"].agg(k="count", mean="mean", std="std").reset_index()
    out["cv_%"] = 100 * out["std"] / out["mean"]
    out["ci_half"] = [t_crit(k - 1) * s / math.sqrt(k) if k > 1 else np.nan for k, s in zip(out["k"], out["std"])]
    out["ci_low"] = out["mean"] - out["ci_half"]
    out["ci_high"] = out["mean"] + out["ci_half"]
    out["method"] = method_cat(out["method"])
    return out.sort_values(keys + ["metric"]).reset_index(drop=True)

perf_summary = summarize(perf_df, ["model", "method"], PERF_KEYS)
perf_summary.round(3)

In [ ]:
# Una métrica por gráfico (escalas distintas: nunca dos ejes Y en el mismo gráfico)
PLOT_PERF = [
    ("prefill_tok_per_s",       "tokens/s", "Prefill throughput"),
    ("decode_tok_per_s",        "tokens/s", "Decode throughput"),
    ("ttft_ms",                 "ms",       "Time to first token"),
    ("perf_peak_mem_device_gb", "GB",       "Memoria pico del dispositivo"),
]

for model in MODELS:
    ps_m = perf_summary[perf_summary["model"] == model]
    if ps_m.empty:
        print(f"{model}: sin datos de rendimiento")
        continue
    fig, axes = plt.subplots(1, len(PLOT_PERF), figsize=(4.2 * len(PLOT_PERF), 4.8))
    for ax, (metric, unit, title) in zip(axes, PLOT_PERF):
        sub = ps_m[ps_m["metric"] == metric].sort_values("method")
        labels = [str(m) for m in sub["method"]]
        x = np.arange(len(sub))
        ax.bar(x, sub["mean"], 0.7, color=[METHOD_COLORS[m] for m in labels])
        err = np.nan_to_num(sub["ci_half"].values)
        ax.errorbar(x, sub["mean"], yerr=[np.minimum(err, sub["mean"].values), err], fmt="none",  # no bajar de 0
                    ecolor="#333333", elinewidth=1.2, capsize=4)
        # cada corrida como un punto, para ver la dispersión real
        for xi, m in zip(x, labels):
            pts = perf_df.loc[(perf_df["model"] == model) & (perf_df["method"] == m), metric].dropna()
            ax.scatter(np.full(len(pts), xi) + np.linspace(-0.12, 0.12, len(pts)), pts, s=18, color="#222222", zorder=3)
        for xi, k in zip(x, sub["k"]):
            ax.text(xi, 0, f"k={k}", ha="center", va="bottom", fontsize=7, color="white")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
        ax.set_ylabel(unit)
        ax.set_title(title, fontsize=10)
        style_ax(ax)
    fig.suptitle(f"{model}: rendimiento, media ± IC 95 % entre corridas (puntos = corridas)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Prefill vs memoria pico, con barras de error (IC 95 %) en los dos ejes
for model in MODELS:
    ps = perf_summary[perf_summary["model"] == model].set_index(["method", "metric"])
    methods_m = [m for m in METHODS
                 if (m, "prefill_tok_per_s") in ps.index and (m, "perf_peak_mem_device_gb") in ps.index]
    if not methods_m:
        continue
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for m in methods_m:
        xm, xe = ps.loc[(m, "prefill_tok_per_s"), ["mean", "ci_half"]]
        ym, ye = ps.loc[(m, "perf_peak_mem_device_gb"), ["mean", "ci_half"]]
        ax.errorbar(xm, ym, xerr=np.nan_to_num(xe), yerr=np.nan_to_num(ye), fmt="o", ms=11,
                    color=METHOD_COLORS[m], ecolor=METHOD_COLORS[m], elinewidth=2, capsize=4,
                    markeredgecolor="white", markeredgewidth=2, label=m, zorder=3)
        ax.annotate(f"{xm:.0f} tok/s\n{ym:.2f} GB", (xm, ym), textcoords="offset points",
                    xytext=(10, 8), fontsize=8, color="#333333")
    ax.set_xlabel("Prefill throughput (tokens/s)")
    ax.set_ylabel("Memoria pico del dispositivo (GB)")
    ax.set_title(f"{model}: prefill vs memoria pico (media ± IC 95 %)")
    ax.margins(0.15)
    style_ax(ax, "both")
    bottom_legend(ax, len(methods_m), y=-0.12)
    plt.tight_layout()
    plt.show()

### Tiempo total de cada benchmark (varianza entre corridas)

In [ ]:
time_summary = summarize(times, ["model", "method", "task"], ["total_time_s"])
display(time_summary.round(2))

for model in MODELS:
    d = time_summary[time_summary["model"] == model]
    tasks = sorted(d["task"].unique())
    fig, ax = plt.subplots(figsize=(11, 5.5))
    grouped_bars(ax, d, "task", "mean", "ci_half", tasks, "Tiempo total (s, escala log)",
                 f"{model}: tiempo por benchmark, media ± IC 95 % entre corridas", fmt="{:.0f}", log=True)
    plt.tight_layout()
    plt.show()

## Notas para la tesis

- **Accuracy:** reportar `valor ± IC95`, con el IC sacado del error estándar binomial (muestreo de preguntas).
  Repetir corridas no lo achica, porque la evaluación es determinista. Lo que lo achicaría es un benchmark con más preguntas.
- **Rendimiento:** reportar media ± IC95 con t de Student e indicar *k* (número de corridas).
- **Comparar métodos:** mirar si el IC de la diferencia contra el baseline cruza el 0 (forest plot).
- **Pendiente:** para tener el IC del F1 de SQAC (y tests pareados) hay que guardar el score de cada pregunta y hacer bootstrap.